<a href="https://colab.research.google.com/github/TediBalint/AI-Jegyzetek/blob/master/Computer%20Vision/K%C3%A9pszegment%C3%A1l%C3%A1s%20Alapok%20UNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Képszegmentálás Alapok: U-Net

A **szegmentálás** pixelszintű osztályozás: minden pixelhez egy osztálycímkét rendelünk.

## Szegmentálás típusok

| Típus | Leírás | Példa |
|-------|--------|-------|
| Semantic | Pixelek osztályozása | Út vs. járda vs. autó |
| Instance | Objektumok szétválasztása | Autó #1, Autó #2 |
| Panoptic | Semantic + Instance | Minden pixel + objektum ID |

## Tartalomjegyzék

1. Szegmentálás alapok
2. U-Net architektúra
3. Skip connections
4. Loss függvények
5. Gyakorlati példa

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
torch.manual_seed(42)

## 1. Szegmentálás alapok

### Kimenet formátum

```
Input:  [B, 3, H, W]     - RGB kép
Output: [B, C, H, W]     - C osztály logit minden pixelre
Mask:   [B, H, W]        - Ground truth (0..C-1 értékek)
```

In [ ]:
def visualize_segmentation_task():
    """Szegmentálás feladat illusztráció."""

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    # Szimulált kép (egyszerű alakzatok)
    np.random.seed(42)
    H, W = 64, 64

    # RGB kép
    image = np.ones((H, W, 3)) * 0.9  # Háttér

    # Kör (piros)
    y, x = np.ogrid[:H, :W]
    circle_mask = ((x - 20)**2 + (y - 25)**2) < 12**2
    image[circle_mask] = [0.8, 0.2, 0.2]

    # Négyzet (kék)
    image[35:55, 35:55] = [0.2, 0.3, 0.8]

    # Ground truth mask
    mask = np.zeros((H, W), dtype=np.int32)
    mask[circle_mask] = 1  # Osztály 1: kör
    mask[35:55, 35:55] = 2  # Osztály 2: négyzet

    # Predicted mask (kis hibákkal)
    pred_mask = mask.copy()
    noise_idx = np.random.choice(H*W, 50, replace=False)
    pred_mask.flat[noise_idx] = np.random.randint(0, 3, 50)

    # Vizualizáció
    axes[0].imshow(image)
    axes[0].set_title('Input kép', fontsize=11)
    axes[0].axis('off')

    im1 = axes[1].imshow(mask, cmap='viridis', vmin=0, vmax=2)
    axes[1].set_title('Ground Truth\n(0=háttér, 1=kör, 2=négyzet)', fontsize=10)
    axes[1].axis('off')

    axes[2].imshow(pred_mask, cmap='viridis', vmin=0, vmax=2)
    axes[2].set_title('Predikció', fontsize=11)
    axes[2].axis('off')

    plt.colorbar(im1, ax=axes[1], ticks=[0, 1, 2], shrink=0.8)
    plt.suptitle('Semantic Segmentation feladat', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_segmentation_task()

## 2. U-Net architektúra

A **U-Net** az encoder-decoder architektúra klasszikusa, skip connection-ökkel.

```
     Encoder (Contracting)              Decoder (Expanding)
    ┌─────────────────┐                ┌─────────────────┐
    │  64 × 64 × 64   │ ─────────────▶ │  64 × 64 × 64   │ → Output
    │       ↓         │   skip conn    │       ↑         │
    │  32 × 32 × 128  │ ─────────────▶ │  32 × 32 × 128  │
    │       ↓         │   skip conn    │       ↑         │
    │  16 × 16 × 256  │ ─────────────▶ │  16 × 16 × 256  │
    │       ↓         │   skip conn    │       ↑         │
    │   8 × 8 × 512   │ ─────────────▶ │   8 × 8 × 512   │
    │       ↓         │                │       ↑         │
    └───────┴─────────┘                └───────┴─────────┘
                  └──▶  Bottleneck  ◀──┘
                         4 × 4 × 1024
```

In [ ]:
class DoubleConv(nn.Module):
    """Két egymást követő Conv-BN-ReLU blokk."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    """U-Net szegmentáláshoz."""
    def __init__(self, in_channels=3, num_classes=2):
        super().__init__()

        # Encoder (downsampling)
        self.enc1 = DoubleConv(in_channels, 64)
        self.enc2 = DoubleConv(64, 128)
        self.enc3 = DoubleConv(128, 256)
        self.enc4 = DoubleConv(256, 512)

        self.pool = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = DoubleConv(512, 1024)

        # Decoder (upsampling)
        self.up4 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.dec4 = DoubleConv(1024, 512)  # 512 + 512 skip

        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = DoubleConv(512, 256)   # 256 + 256 skip

        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = DoubleConv(256, 128)   # 128 + 128 skip

        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = DoubleConv(128, 64)    # 64 + 64 skip

        # Output
        self.out_conv = nn.Conv2d(64, num_classes, 1)

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        # Bottleneck
        b = self.bottleneck(self.pool(e4))

        # Decoder + Skip connections
        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))

        return self.out_conv(d1)

# Teszt
model = UNet(in_channels=3, num_classes=5)
x = torch.randn(2, 3, 128, 128)
out = model(x)
print(f"Input: {x.shape}")
print(f"Output: {out.shape}  (5 osztály logit minden pixelre)")

## 3. Skip Connections

### Miért fontosak?

Az encoder **elveszíti a finom részleteket** a pooling miatt. A skip connection visszahozza ezeket!

In [ ]:
def visualize_skip_connection_effect():
    """Skip connection hatásának vizualizációja."""

    fig, axes = plt.subplots(2, 4, figsize=(14, 7))

    # Eredeti "kép" (élek, részletek)
    np.random.seed(42)
    H, W = 64, 64

    # Részletes minta
    x = np.linspace(0, 4*np.pi, W)
    y = np.linspace(0, 4*np.pi, H)
    X, Y = np.meshgrid(x, y)
    original = np.sin(X) * np.cos(Y) + 0.5*np.sin(3*X) * np.cos(3*Y)

    # Encoder szintek (downsampling + info loss)
    from scipy.ndimage import zoom

    enc_sizes = [64, 32, 16, 8]
    encoder_features = []

    for i, size in enumerate(enc_sizes):
        factor = size / 64
        downsampled = zoom(original, factor, order=1)
        encoder_features.append(downsampled)
        axes[0, i].imshow(downsampled, cmap='viridis')
        axes[0, i].set_title(f'Encoder {i+1}\n{size}×{size}', fontsize=10)
        axes[0, i].axis('off')

    # Decoder (upsampling - WITH vs WITHOUT skip)
    # Bottleneck-ből indulunk
    bottleneck = zoom(original, 4/64, order=1)  # 4x4

    # Skip nélkül
    no_skip = zoom(bottleneck, 16, order=1)  # Back to 64x64

    # Skip-pel (szimulált)
    with_skip = 0.3 * zoom(bottleneck, 16, order=1) + 0.7 * original

    axes[1, 0].imshow(bottleneck, cmap='viridis')
    axes[1, 0].set_title('Bottleneck\n4×4', fontsize=10)
    axes[1, 0].axis('off')

    axes[1, 1].imshow(no_skip, cmap='viridis')
    axes[1, 1].set_title('Skip NÉLKÜL\n(elmosódott)', fontsize=10)
    axes[1, 1].axis('off')

    axes[1, 2].imshow(with_skip, cmap='viridis')
    axes[1, 2].set_title('Skip-PEL\n(részletek megmaradnak)', fontsize=10)
    axes[1, 2].axis('off')

    axes[1, 3].imshow(original, cmap='viridis')
    axes[1, 3].set_title('Eredeti\n(cél)', fontsize=10)
    axes[1, 3].axis('off')

    axes[0, 0].set_ylabel('Encoder\n(downsampling)', fontsize=11)
    axes[1, 0].set_ylabel('Decoder\n(upsampling)', fontsize=11)

    plt.suptitle('Skip Connection: Finom részletek megőrzése', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

try:
    from scipy.ndimage import zoom
    visualize_skip_connection_effect()
except ImportError:
    print("scipy szükséges a vizualizációhoz")

## 4. Loss függvények

### Cross-Entropy Loss

Alap választás multi-class szegmentáláshoz:

$$\mathcal{L}_{CE} = -\frac{1}{HW}\sum_{i,j} \sum_{c} y_{ijc} \log(\hat{y}_{ijc})$$

### Dice Loss

IoU-szerű metrika, jobb kiegyensúlyozatlan osztályoknál:

$$\text{Dice} = \frac{2|A \cap B|}{|A| + |B|}$$

$$\mathcal{L}_{Dice} = 1 - \text{Dice}$$

In [ ]:
def dice_loss(pred, target, smooth=1e-5):
    """
    Dice loss szegmentáláshoz.

    pred: [B, C, H, W] - softmax után
    target: [B, H, W] - osztály indexek
    """
    num_classes = pred.shape[1]

    # One-hot encoding
    target_onehot = F.one_hot(target, num_classes).permute(0, 3, 1, 2).float()

    # Flatten
    pred_flat = pred.view(pred.shape[0], num_classes, -1)
    target_flat = target_onehot.view(target_onehot.shape[0], num_classes, -1)

    # Dice per class
    intersection = (pred_flat * target_flat).sum(dim=2)
    union = pred_flat.sum(dim=2) + target_flat.sum(dim=2)

    dice = (2 * intersection + smooth) / (union + smooth)

    return 1 - dice.mean()


def combined_loss(pred, target, alpha=0.5):
    """CE + Dice kombinált loss."""
    ce = F.cross_entropy(pred, target)
    pred_soft = F.softmax(pred, dim=1)
    dice = dice_loss(pred_soft, target)
    return alpha * ce + (1 - alpha) * dice


# Teszt
pred = torch.randn(2, 3, 32, 32)  # 3 osztály
target = torch.randint(0, 3, (2, 32, 32))

ce_loss = F.cross_entropy(pred, target)
d_loss = dice_loss(F.softmax(pred, dim=1), target)
comb_loss = combined_loss(pred, target)

print(f"Cross-Entropy: {ce_loss:.4f}")
print(f"Dice Loss: {d_loss:.4f}")
print(f"Combined: {comb_loss:.4f}")

In [ ]:
# IoU (Intersection over Union) metrika
def iou_score(pred, target, num_classes):
    """
    Mean IoU számítása.

    pred: [B, H, W] - predikált osztályok
    target: [B, H, W] - ground truth
    """
    ious = []

    for c in range(num_classes):
        pred_c = (pred == c)
        target_c = (target == c)

        intersection = (pred_c & target_c).sum().float()
        union = (pred_c | target_c).sum().float()

        if union > 0:
            ious.append(intersection / union)

    return torch.stack(ious).mean()

# Vizualizáció
def visualize_iou():
    fig, ax = plt.subplots(figsize=(8, 4))

    # Két halmaz
    theta = np.linspace(0, 2*np.pi, 100)

    # Pred (kék)
    x1, y1 = np.cos(theta) - 0.3, np.sin(theta)
    ax.fill(x1, y1, alpha=0.4, color='blue', label='Predikció')

    # Target (piros)
    x2, y2 = np.cos(theta) + 0.3, np.sin(theta)
    ax.fill(x2, y2, alpha=0.4, color='red', label='Ground Truth')

    # Intersection (zöld)
    ax.fill_between([-0.3, 0.3], [-1, -1], [1, 1], alpha=0.3, color='green', label='Intersection')

    ax.set_xlim(-2, 2)
    ax.set_ylim(-1.5, 1.5)
    ax.set_aspect('equal')
    ax.legend()
    ax.set_title('IoU = Intersection / Union', fontsize=12)

    plt.tight_layout()
    plt.show()

visualize_iou()

## 5. Gyakorlati példa

Egyszerű szegmentálási feladat: geometriai alakzatok szétválasztása.

In [ ]:
def create_segmentation_dataset(n_samples=100, img_size=64):
    """Szintetikus szegmentálási adathalmaz."""
    images = []
    masks = []

    for _ in range(n_samples):
        img = np.ones((img_size, img_size, 3)) * 0.9
        mask = np.zeros((img_size, img_size), dtype=np.int64)

        # Random kör
        cx, cy = np.random.randint(15, img_size-15, 2)
        r = np.random.randint(8, 15)
        y, x = np.ogrid[:img_size, :img_size]
        circle = ((x - cx)**2 + (y - cy)**2) < r**2
        img[circle] = [0.8, 0.2, 0.2]
        mask[circle] = 1

        # Random négyzet
        sx, sy = np.random.randint(5, img_size-20, 2)
        size = np.random.randint(10, 18)
        img[sy:sy+size, sx:sx+size] = [0.2, 0.3, 0.8]
        mask[sy:sy+size, sx:sx+size] = 2

        images.append(img.transpose(2, 0, 1))  # CHW
        masks.append(mask)

    return torch.FloatTensor(np.array(images)), torch.LongTensor(np.array(masks))

# Adathalmaz
X_train, y_train = create_segmentation_dataset(200)
X_test, y_test = create_segmentation_dataset(20)

print(f"Train: {X_train.shape}, {y_train.shape}")
print(f"Test: {X_test.shape}, {y_test.shape}")

# Minta vizualizáció
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i in range(5):
    axes[0, i].imshow(X_train[i].permute(1, 2, 0))
    axes[0, i].axis('off')
    axes[1, i].imshow(y_train[i], cmap='viridis', vmin=0, vmax=2)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Kép')
axes[1, 0].set_ylabel('Mask')
plt.suptitle('Szintetikus szegmentálási adatok', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Kis U-Net tanítása
class TinyUNet(nn.Module):
    """Kisebb U-Net a demóhoz."""
    def __init__(self, num_classes=3):
        super().__init__()

        self.enc1 = nn.Sequential(nn.Conv2d(3, 32, 3, 1, 1), nn.ReLU())
        self.enc2 = nn.Sequential(nn.Conv2d(32, 64, 3, 2, 1), nn.ReLU())
        self.enc3 = nn.Sequential(nn.Conv2d(64, 128, 3, 2, 1), nn.ReLU())

        self.dec3 = nn.Sequential(nn.ConvTranspose2d(128, 64, 2, 2), nn.ReLU())
        self.dec2 = nn.Sequential(nn.Conv2d(128, 64, 3, 1, 1), nn.ReLU())
        self.dec1 = nn.Sequential(nn.ConvTranspose2d(64, 32, 2, 2), nn.ReLU())
        self.out = nn.Conv2d(64, num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)

        d3 = self.dec3(e3)
        d2 = self.dec2(torch.cat([d3, e2], dim=1))
        d1 = self.dec1(d2)
        return self.out(torch.cat([d1, e1], dim=1))

# Tanítás
model = TinyUNet(num_classes=3)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

losses = []
for epoch in range(30):
    model.train()

    # Mini-batch
    idx = torch.randperm(len(X_train))[:32]
    x_batch, y_batch = X_train[idx], y_train[idx]

    pred = model(x_batch)
    loss = combined_loss(pred, y_batch)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}: Loss = {loss.item():.4f}")

plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Tanítási loss')
plt.show()

In [ ]:
# Eredmények
model.eval()
with torch.no_grad():
    pred = model(X_test[:5])
    pred_mask = pred.argmax(dim=1)

fig, axes = plt.subplots(3, 5, figsize=(12, 7))
for i in range(5):
    axes[0, i].imshow(X_test[i].permute(1, 2, 0))
    axes[0, i].axis('off')

    axes[1, i].imshow(y_test[i], cmap='viridis', vmin=0, vmax=2)
    axes[1, i].axis('off')

    axes[2, i].imshow(pred_mask[i], cmap='viridis', vmin=0, vmax=2)
    axes[2, i].axis('off')

axes[0, 0].set_ylabel('Input', fontsize=11)
axes[1, 0].set_ylabel('GT Mask', fontsize=11)
axes[2, 0].set_ylabel('Predikció', fontsize=11)

plt.suptitle('U-Net szegmentálás eredmények', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# IoU
miou = iou_score(pred_mask, y_test[:5], num_classes=3)
print(f"\nMean IoU: {miou:.4f}")

## Összefoglalás

### U-Net kulcs elemei

1. **Encoder**: Downsampling → szemantikus információ
2. **Decoder**: Upsampling → eredeti felbontás
3. **Skip connections**: Finom részletek megőrzése

### Loss függvények

| Loss | Előny | Hátrány |
|------|-------|----------|
| Cross-Entropy | Egyszerű | Kiegyensúlyozatlanságra érzékeny |
| Dice | IoU-szerű | Instabil kis régiókra |
| Focal | Nehéz példákra fókuszál | Több hiperparaméter |

### Metrikák

- **Pixel Accuracy**: Helyes pixelek aránya
- **Mean IoU**: Osztályonkénti IoU átlaga
- **Dice Score**: 2 × |A∩B| / (|A| + |B|)